# Effect of Activation Functions on Neural Network Performance

## Problem Statement
In this exercise, you will investigate how different activation functions affect neural network training and performance. You will:
1. Train three separate neural networks on the **FashionMNIST** dataset, each using a different activation function: **ReLU**, **Tanh**, and **GELU**
2. Compare convergence speed and final accuracy across all three models
3. Visualize training curves (loss and accuracy) for each activation function
4. Analyze and explain which activation function performs best and why

## Step 1: Data Acquisition
**Dataset**: FashionMNIST
- **Source**: Available through `torchvision.datasets.FashionMNIST`
- **Description**: 70,000 grayscale images (28x28 pixels) of 10 fashion categories
- **Classes**: T-shirt/top, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle boot
- **Split**: 60,000 training images, 10,000 test images

**Task**: Load the FashionMNIST dataset and create DataLoaders for training and testing.

In [1]:
# Import necessary libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader
import time

In [2]:
# Load FashionMNIST dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = torchvision.datasets.FashionMNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.FashionMNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

Processing...
Done!


## Step 2: Data Wrangling
**Preprocessing Requirements**:
- Convert images to tensors
- Normalize pixel values to have mean=0.5 and std=0.5
- Use batch size of 64 for training
- Shuffle training data

**Task**: Implement data preprocessing and verify data shapes.

In [3]:
# Verify data shapes and visualize sample images
classes = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
           'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

# Get a batch of training data
dataiter = iter(train_loader)
images, labels = next(dataiter)

print(f"Training set size: {len(train_dataset)}")
print(f"Test set size: {len(test_dataset)}")
print(f"Batch shape: {images.shape}")
print(f"Label shape: {labels.shape}")
print(f"\nClass labels:")
for i, cls in enumerate(classes):
    print(f"{i}: {cls}")

Training set size: 60000
Test set size: 10000
Batch shape: torch.Size([64, 1, 28, 28])
Label shape: torch.Size([64])

Class labels:
0: T-shirt/top
1: Trouser
2: Pullover
3: Dress
4: Coat
5: Sandal
6: Shirt
7: Sneaker
8: Bag
9: Ankle boot


In [4]:
# Visualize sample images
fig, axes = plt.subplots(1, 5, figsize=(12, 3))
for i in range(5):
    axes[i].imshow(images[i].squeeze(), cmap='gray')
    axes[i].set_title(classes[labels[i]])
    axes[i].axis('off')
plt.tight_layout()
plt.show()

## Step 3: Model Implementation
**Architecture Specification**:
- Input: 28x28 grayscale images (flattened to 784 features)
- Hidden Layer 1: 512 units + Activation
- Hidden Layer 2: 256 units + Activation
- Hidden Layer 3: 128 units + Activation
- Output Layer: 10 units (no activation, will use CrossEntropyLoss)

**Task**: Create a flexible neural network class that accepts activation function as a parameter.

In [5]:
# Define Neural Network with configurable activation
class NeuralNet(nn.Module):
    def __init__(self, activation_fn):
        super(NeuralNet, self).__init__()
        self.fc1 = nn.Linear(28 * 28, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, 10)
        self.activation = activation_fn
    
    def forward(self, x):
        x = x.view(x.size(0), -1)  # Flatten
        x = self.activation(self.fc1(x))
        x = self.activation(self.fc2(x))
        x = self.activation(self.fc3(x))
        x = self.fc4(x)
        return x

## Step 4: Training
**Training Configuration**:
- **Loss Function**: CrossEntropyLoss
- **Optimizer**: Adam with learning rate 0.001
- **Epochs**: 20
- **Activation Functions to Compare**: ReLU, Tanh, GELU

**Task**: Implement training loop that records loss and accuracy for each epoch.

In [6]:
# Training function
def train_model(model, train_loader, test_loader, criterion, optimizer, epochs, device):
    train_losses = []
    train_accuracies = []
    test_losses = []
    test_accuracies = []
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        
        train_loss = running_loss / len(train_loader)
        train_acc = 100 * correct / total
        train_losses.append(train_loss)
        train_accuracies.append(train_acc)
        
        # Evaluation phase
        model.eval()
        test_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                test_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        test_loss = test_loss / len(test_loader)
        test_acc = 100 * correct / total
        test_losses.append(test_loss)
        test_accuracies.append(test_acc)
        
        print(f"Epoch [{epoch+1}/{epochs}], "
              f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%, "
              f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%")
    
    return {
        'train_losses': train_losses,
        'train_accuracies': train_accuracies,
        'test_losses': test_losses,
        'test_accuracies': test_accuracies
    }

In [7]:
# Train models with different activations
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Define activation functions
activations = {
    'ReLU': nn.ReLU(),
    'Tanh': nn.Tanh(),
    'GELU': nn.GELU()
}

results = {}
epochs = 20

for name, activation_fn in activations.items():
    print(f"\n{'='*40}")
    print(f"Training with {name} activation")
    print(f"{'='*40}")
    
    model = NeuralNet(activation_fn).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    history = train_model(model, train_loader, test_loader, criterion, optimizer, epochs, device)
    results[name] = history

Using device: cuda

Training with ReLU activation
Epoch [1/20], Train Loss: 0.5012, Train Acc: 82.14%, Test Loss: 0.4234, Test Acc: 84.68%
Epoch [2/20], Train Loss: 0.3654, Train Acc: 86.78%, Test Loss: 0.3789, Test Acc: 86.42%
Epoch [3/20], Train Loss: 0.3234, Train Acc: 88.23%, Test Loss: 0.3456, Test Acc: 87.54%
Epoch [4/20], Train Loss: 0.2956, Train Acc: 89.12%, Test Loss: 0.3287, Test Acc: 88.12%
Epoch [5/20], Train Loss: 0.2745, Train Acc: 89.89%, Test Loss: 0.3198, Test Acc: 88.45%
Epoch [6/20], Train Loss: 0.2567, Train Acc: 90.54%, Test Loss: 0.3145, Test Acc: 88.67%
Epoch [7/20], Train Loss: 0.2412, Train Acc: 91.08%, Test Loss: 0.3089, Test Acc: 88.92%
Epoch [8/20], Train Loss: 0.2278, Train Acc: 91.54%, Test Loss: 0.3067, Test Acc: 89.01%
Epoch [9/20], Train Loss: 0.2156, Train Acc: 91.98%, Test Loss: 0.3054, Test Acc: 89.15%
Epoch [10/20], Train Loss: 0.2045, Train Acc: 92.38%, Test Loss: 0.3045, Test Acc: 89.23%
Epoch [11/20], Train Loss: 0.1943, Train Acc: 92.75%, Test 

## Step 5: Evaluation and Analysis
**Metrics to Compare**:
1. Final test accuracy
2. Training convergence speed (epochs to reach 85% accuracy)
3. Training stability (smoothness of curves)

**Visualizations Required**:
- Training loss curves for all three activations
- Test accuracy curves for all three activations
- Comparison table of final metrics

**Task**: Plot training curves and analyze results.

In [8]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot training loss
for name, history in results.items():
    ax1.plot(history['train_losses'], label=name, linewidth=2)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Training Loss', fontsize=12)
ax1.set_title('Training Loss Comparison', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Plot test accuracy
for name, history in results.items():
    ax2.plot(history['test_accuracies'], label=name, linewidth=2)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Test Accuracy (%)', fontsize=12)
ax2.set_title('Test Accuracy Comparison', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [9]:
# Compare final metrics
print("\n" + "="*40)
print("Final Performance Comparison")
print("="*40 + "\n")

print(f"{'Activation':<11}| {'Final Test Acc':<15}| {'Epochs to 85%':<14}| {'Final Train Loss'}")
print("-" * 11 + "|" + "-" * 16 + "|" + "-" * 15 + "|" + "-" * 18)

best_acc = 0
best_activation = ""
fastest_convergence = float('inf')
fastest_activation = ""

for name, history in results.items():
    final_acc = history['test_accuracies'][-1]
    final_loss = history['train_losses'][-1]
    
    # Find epoch when test accuracy first reached 85%
    epochs_to_85 = next((i+1 for i, acc in enumerate(history['test_accuracies']) if acc >= 85), 'N/A')
    
    print(f"{name:<11}|     {final_acc:.2f}%     |       {epochs_to_85}       |      {final_loss:.4f}")
    
    if final_acc > best_acc:
        best_acc = final_acc
        best_activation = name
    
    if epochs_to_85 != 'N/A' and epochs_to_85 < fastest_convergence:
        fastest_convergence = epochs_to_85
        fastest_activation = name

print(f"\nBest performing activation: {best_activation} with {best_acc:.2f}% test accuracy")
print(f"Fastest convergence: {fastest_activation} reached 85% in {fastest_convergence} epoch(s)")


Final Performance Comparison

Activation | Final Test Acc | Epochs to 85% | Final Train Loss
-----------|----------------|---------------|------------------
ReLU       |     89.71%     |       2       |      0.1289
Tanh       |     90.00%     |       3       |      0.2207
GELU       |     91.17%     |       1       |      0.1439

Best performing activation: GELU with 91.17% test accuracy
Fastest convergence: GELU reached 85% in 1 epoch(s)


## Analysis and Conclusions

### 1. Which activation function achieved the highest final accuracy?

**GELU achieved the highest final test accuracy at 91.17%**, followed by Tanh at 90.00% and ReLU at 89.71%. This represents approximately a 1.5% improvement over ReLU.

### 2. Which activation function converged the fastest?

**GELU converged the fastest**, reaching 85% test accuracy in just 1 epoch, compared to 2 epochs for ReLU and 3 epochs for Tanh. This demonstrates GELU's superior learning dynamics in the early stages of training.

### 3. Which activation function had the smoothest training curves?

**GELU exhibited the smoothest training curves**, with consistent improvements and minimal fluctuations. Both ReLU and Tanh showed slightly more variation, particularly in the later epochs where overfitting began to occur (train loss decreased while test loss plateaued).

### 4. Explain why certain activation functions performed better:

**ReLU (Rectified Linear Unit)**:
- **Formula**: f(x) = max(0, x)
- **Advantages**: Computationally efficient, helps mitigate vanishing gradient problem, sparse activation
- **Limitations**: Can suffer from "dying ReLU" problem where neurons output zero for all inputs, has a hard zero boundary
- **Performance**: Achieved good results (89.71%) but was outperformed by GELU, likely due to its non-smooth activation at x=0

**Tanh (Hyperbolic Tangent)**:
- **Formula**: f(x) = (e^x - e^(-x)) / (e^x + e^(-x))
- **Advantages**: Zero-centered outputs (range: -1 to 1), smooth and differentiable everywhere
- **Limitations**: Can suffer from vanishing gradients for large input values, saturation at extremes
- **Performance**: Performed well (90.00%), benefiting from zero-centered outputs which help with gradient flow, but converged slower than GELU

**GELU (Gaussian Error Linear Unit)**:
- **Formula**: f(x) = x * Φ(x), where Φ is the cumulative distribution function of the standard normal distribution
- **Advantages**: Smooth, non-monotonic, stochastic regularization effect, better gradient flow
- **Modern usage**: Widely used in transformer architectures (BERT, GPT) and modern deep learning
- **Performance**: Best overall performance (91.17%), combining the benefits of smoothness and probabilistic gating

### 5. Key differences between these activation functions:

**Mathematical Formulation**:
- ReLU is piecewise linear with a hard boundary at zero
- Tanh is a sigmoid-shaped function with smooth transitions
- GELU combines linearity with probabilistic weighting based on the normal distribution

**Gradient Behavior**:
- ReLU: Gradient is either 0 or 1 (binary)
- Tanh: Gradient decreases as values move away from zero (vanishing gradient risk)
- GELU: Smooth gradients throughout with better flow properties

**Common Use Cases**:
- ReLU: Default choice for CNNs, general deep learning tasks
- Tanh: RNNs, specific architectures requiring zero-centered outputs
- GELU: Modern transformers, state-of-the-art NLP models, complex vision tasks

### Conclusion

For the FashionMNIST classification task, **GELU activation demonstrated superior performance** across all metrics. It achieved the highest accuracy (91.17%), fastest convergence (1 epoch to 85%), and smoothest training dynamics. This aligns with recent trends in deep learning where GELU has become increasingly popular, especially in transformer-based architectures.

The results suggest that for similar image classification tasks, GELU should be preferred over traditional activations like ReLU and Tanh, despite its slightly higher computational cost. The performance gains justify the additional complexity, particularly in production systems where model accuracy is critical.